# Lab 10 — Enterprise Deployment: Middleware, Circuit Breakers & Cloud IaC

**Prerequisite:** Lab 3 (Production Deployment) and Lab 5 (Security). Those gave you a FastAPI service with sessions, rate limiting, and guardrails.  
This lab adds what an **enterprise** demands on top of that:

1. **Budget middleware** — a hard spend cap so a runaway loop can't generate a $10,000 bill
2. **Audit logging** — an immutable, PII-free compliance trail (GDPR/HIPAA)
3. **Circuit breaker** — fail fast when the upstream LLM API is down
4. **Cloud IaC** — the same container deployed to AWS, GCP, or Azure

The composable-middleware idea is the senior insight here: each concern is one wrapper, stacked in a deliberate order.

> **Tested on:** `gpt-4o-mini`.  
> **Cost:** ~$0.01–0.05. The middleware stack and circuit-breaker demos make only a handful of short LLM calls (most requests are intentionally blocked by budget, scrubbing, or the open circuit, so they never reach the API). The cloud IaC section is read-only — no API or cloud spend.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from enterprise_middleware import BudgetMiddleware, PIIScrubber, AuditLogger, BudgetExceededError
from circuit_breaker import CircuitBreaker, CircuitBreakerError
from openai import OpenAI
client = OpenAI()

def base_agent(prompt: str, **kwargs) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=80,
    )
    return resp.choices[0].message.content

print('Lab 10 ready ✓')

## 1. The middleware stack — order matters

```
Request
  → AuditLogger        (log every attempt, even rejected ones)
  → PIIScrubber        (strip PII BEFORE it reaches the LLM — GDPR Art. 25)
  → BudgetMiddleware   (hard-stop before spending)
  → CircuitBreaker     (fail fast if API is down)
  → LLM
```

Why this order? Audit is outermost so it records rejections too. PII scrubbing must happen before *anything* logs or sends the text. Budget check is cheap, so do it before the network call. The circuit breaker sits closest to the fragile dependency.

In [ ]:
import tempfile
log_path = os.path.join(tempfile.gettempdir(), 'lab10_audit.jsonl')
if os.path.exists(log_path):
    os.remove(log_path)

protected = CircuitBreaker(base_agent, failure_threshold=2, fallback=lambda p, **k: 'Service unavailable.')
agent = AuditLogger(
    PIIScrubber(
        BudgetMiddleware(protected, global_budget_usd=0.02, per_user_budget_usd=0.02)
    ),
    log_file=log_path,
)
print('Stacked: AuditLogger → PIIScrubber → BudgetMiddleware → CircuitBreaker → LLM')

In [ ]:
queries = [
    ('alice', 'What are the benefits of containerisation?'),
    ('alice', 'My email is alice@corp.com and SSN 123-45-6789 — help me reset my account.'),
    ('bob',   'Explain circuit breakers in one sentence.'),
]
for user_id, q in queries:
    try:
        r = agent(q, user_id=user_id)
        print(f'[{user_id}] {r[:70]}')
    except BudgetExceededError as e:
        print(f'[{user_id}] BUDGET STOP: {e}')

## 2. Verify the compliance trail
The audit log stores a **hash** of each prompt (never raw PII) plus a short response preview — enough to prove what happened in a regulatory review without storing personal data.

In [ ]:
import json
print('Audit entries:')
with open(log_path) as f:
    for line in f:
        e = json.loads(line)
        print(f"  {e['entry_id'][:8]}  user={e['user_id']:6}  hash={e['prompt_hash']}  lat={e['latency_seconds']:.2f}s")

print('\nCompliance summary:')
print(json.dumps(agent.compliance_summary(), indent=2))
print('\nPII redactions made:', agent.agent_fn.redaction_count)

## 3. Circuit breaker — fail fast during an outage
If the LLM API goes down, a naive agent queues thousands of slow-failing requests, exhausting threads and budget. The breaker trips after N failures and fails instantly (or serves a fallback) until a recovery window passes.

In [ ]:
fail_window = range(3, 6)  # simulate outage on calls 3-5
n = [0]
def flaky(prompt: str, **kw) -> str:
    n[0] += 1
    if n[0] in fail_window:
        raise ConnectionError(f'API down (call {n[0]})')
    return base_agent(prompt)

cb = CircuitBreaker(flaky, failure_threshold=2, recovery_timeout=1.0,
                    fallback=lambda p, **k: '[fallback] try again shortly')
import time
for i in range(8):
    try:
        out = cb(f'Q{i+1}')
        print(f'[{i+1}] {cb.state.value:10} {out[:40]}')
    except Exception as e:
        print(f'[{i+1}] {cb.state.value:10} ERR {e}')
    if i == 5:
        time.sleep(1.1)  # let recovery_timeout elapse

print('\nStats:', cb.stats())

## 4. Cloud IaC — same container, three clouds
Lab 3 gave you the Dockerfile. The `deploy/` folder holds Infrastructure-as-Code to run that container on each major cloud. Read them — you don't need accounts to understand the shape.

In [ ]:
options = {
    'AWS Lambda + API Gateway': ('deploy/aws_lambda.tf', 'pay-per-request, 15min cap, cold starts; Terraform'),
    'GCP Cloud Run':            ('deploy/gcp_cloudrun.yaml', 'scale-to-zero, persistent HTTP, 60min; single YAML'),
    'Azure Container Apps':     ('deploy/azure_container_apps.bicep', 'KEDA scaling, DAPR, microservices; Bicep'),
}
for name, (path, note) in options.items():
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'{name:28} {path:36} ({size} B) — {note}')

print('\nDecision rule:')
print('  Bursty + already on AWS  → Lambda')
print('  HTTP agent, want simple  → Cloud Run')
print('  Microsoft enterprise     → Container Apps')

## Exercise — the full production stack
Wire **Lab 3's FastAPI server** together with this lab's middleware and Lab 5's guardrails:
1. Stack: `AuditLogger → PIIScrubber → Guardrails(Lab 5) → BudgetMiddleware → CircuitBreaker → agent`
2. Send 10 requests: 2 with PII, 1 injection attempt, 1 that exceeds budget, 1 during a simulated outage
3. Print the audit log, budget report, guardrail blocks, and circuit-breaker stats

This is the complete shippable system — the thing you describe end-to-end in a system-design interview.

In [ ]:
# Your code here — the complete production stack
